In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator

# ---------------------------------------------------------
# 1. PREPARE DATA
# ---------------------------------------------------------
# We use standard MNIST data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

# Standard PyTorch DataLoader
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

# ---------------------------------------------------------
# 2. DEFINE MODEL
# ---------------------------------------------------------
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Note: We use GroupNorm instead of BatchNorm because 
        # BatchNorm does not work well with Differential Privacy
        self.conv1 = nn.Conv2d(1, 16, 8, 2, padding=3)
        self.act1 = nn.ReLU()
        self.pool1 = nn.AvgPool2d(2, 1)
        self.conv2 = nn.Conv2d(16, 32, 4, 2)
        self.act2 = nn.ReLU()
        self.pool2 = nn.AvgPool2d(2, 1)
        self.fc1 = nn.Linear(32 * 4 * 4, 32)
        self.act3 = nn.ReLU()
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        x = self.act1(self.conv1(x))
        x = self.pool1(x)
        x = self.act2(self.conv2(x))
        x = self.pool2(x)
        x = torch.flatten(x, 1)
        x = self.act3(self.fc1(x))
        x = self.fc2(x)
        return x

model = Net()

# Important: Validate that the model layers are compatible with DP
# (e.g., replaces BatchNorm with GroupNorm if necessary)
model = ModuleValidator.fix(model)

# ---------------------------------------------------------
# 3. DEFINE OPTIMIZER & LOSS
# ---------------------------------------------------------
optimizer = optim.SGD(model.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

# ---------------------------------------------------------
# 4. ENTER PRIVACY ENGINE (Your requested part)
# ---------------------------------------------------------
privacy_engine = PrivacyEngine()

# This wraps the model and optimizer to handle:
# 1. Gradient Clipping (max_grad_norm)
# 2. Noise Addition (noise_multiplier)
model, optimizer, train_loader = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader,
    noise_multiplier=1.1,  # Standard deviation of noise added to gradients
    max_grad_norm=1.0,     # Clip gradients to this maximum norm
)

print(f"Privacy Engine attached. Training with Differential Privacy...")

# ---------------------------------------------------------
# 5. TRAINING LOOP
# ---------------------------------------------------------
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        
        # The 'step' here is now a PRIVATE step (clips grad + adds noise)
        optimizer.step() 

        if batch_idx % 100 == 0:
            print(f'Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}]\tLoss: {loss.item():.6f}')

# ---------------------------------------------------------
# 6. EVALUATION & PRIVACY ACCOUNTING
# ---------------------------------------------------------
# We calculate privacy budget (epsilon)
# delta is usually set to 1 / len(dataset)
DELTA = 1e-5 

for epoch in range(1, 3): # Run for 2 epochs for demonstration
    train(model, "cpu", train_loader, optimizer, epoch)
    
    # Calculate how much privacy budget we have spent so far
    epsilon = privacy_engine.get_epsilon(DELTA)
    
    print(f"\nResult after Epoch {epoch}:")
    print(f"  Example Privacy Limit (Epsilon): {epsilon:.2f}")
    print(f"  (This guarantees the model remains differentially private with these parameters)\n")

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from torch.utils.data import Subset # <--- Added this

# 1. PREPARE DATA
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

# Download full dataset
full_train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)

# --- CHANGE #1: USE TINY SUBSET ---
# We only take the first 256 images. 
# This ensures the epoch finishes in seconds.
subset_indices = list(range(256))
small_dataset = Subset(full_train_dataset, subset_indices)

train_loader = torch.utils.data.DataLoader(small_dataset, batch_size=32, shuffle=True)

# 2. DEFINE MODEL
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 8, 2, padding=3)
        self.act1 = nn.ReLU()
        self.pool1 = nn.AvgPool2d(2, 1)
        self.conv2 = nn.Conv2d(16, 32, 4, 2)
        self.act2 = nn.ReLU()
        self.pool2 = nn.AvgPool2d(2, 1)
        self.fc1 = nn.Linear(32 * 4 * 4, 32)
        self.act3 = nn.ReLU()
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        x = self.act1(self.conv1(x))
        x = self.pool1(x)
        x = self.act2(self.conv2(x))
        x = self.pool2(x)
        x = torch.flatten(x, 1)
        x = self.act3(self.fc1(x))
        x = self.fc2(x)
        return x

model = Net()
model = ModuleValidator.fix(model)
optimizer = optim.SGD(model.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

# 3. PRIVACY ENGINE
privacy_engine = PrivacyEngine()

model, optimizer, train_loader = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader,
    noise_multiplier=1.1,
    max_grad_norm=1.0,
)

print(f"DEBUG MODE: Training on {len(subset_indices)} images only...")

# 4. TRAINING LOOP
def train(model, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        # --- CHANGE #2: PRINT EVERY BATCH ---
        # We print every single step so you know it's not frozen
        print(f'Epoch: {epoch} [Batch {batch_idx+1}/{len(train_loader)}]\tLoss: {loss.item():.6f}')

# 5. EXECUTION
# --- CHANGE #3: ONE EPOCH ONLY ---
DELTA = 1e-5

train(model, train_loader, optimizer, 1)

epsilon = privacy_engine.get_epsilon(DELTA)
print(f"\nSUCCESS! Privacy Mechanism is working.")
print(f"Epsilon spent: {epsilon:.2f}")

100.0%
100.0%
100.0%
100.0%
/home/valer/venvs/opacus_venv/lib/python3.11/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


DEBUG MODE: Training on 256 images only...
Epoch: 1 [Batch 1/8]	Loss: 2.313113
Epoch: 1 [Batch 2/8]	Loss: 2.327081
Epoch: 1 [Batch 3/8]	Loss: 2.319593
Epoch: 1 [Batch 4/8]	Loss: 2.312907
Epoch: 1 [Batch 5/8]	Loss: 2.303945


/tmp/ipykernel_1399/349018949.py:75: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


Epoch: 1 [Batch 6/8]	Loss: 2.315080
Epoch: 1 [Batch 7/8]	Loss: 2.309405
Epoch: 1 [Batch 8/8]	Loss: 2.329197

SUCCESS! Privacy Mechanism is working.
Epsilon spent: 2.65
